In [63]:
import pandas as pd
import numpy as np
import json
import pandas as pd
import re
import unicodedata
import pandas as pd
from collections import defaultdict

In [64]:
# %% CELL 1 — Imports, config, load and filter to playing squad

FANTASY_CSV   = "fantasy_optimizer.csv"
STATS_CSV     = "players_data-2025_2026.csv"
OUT_CSV       = "fantasy_enriched.csv"
 
MAX_EDIT_DIST = 1
 
MERGE_COUNT_COLS = [
    "MP", "Starts", "Min", "90s",
    "Gls", "Ast", "G+A", "xG", "xAG", "npxG", "G-PK",
    "Tkl", "TklW", "Blocks", "Int", "Tkl+Int", "Clr", "Err",
    "PrgP", "PrgC", "KP", "PPA", "xA", "Ast_stats_passing",
    "GA", "Saves", "CS", "PKA", "PKsv",
    "Touches", "Carries", "PrgR", "Mis", "Dis",
    "CrdY", "CrdR", "PKwon", "PKcon", "Recov",
    "PK", "PKatt", "SoTA", "PKm",
    "Sh", "SoT",
    "Fls", "Fld", "Off", "Crs",
    "OG", "2CrdY",
]
MERGE_RATE_COLS = [
    "Cmp%_stats_passing",
    "Save%",
    "CS%",
    "GA90", "SoT%", "Sh/90", "SoT/90", "G/Sh", "G/SoT",
]
MERGE_META_COLS = ["Pos", "Squad", "Comp", "Age"]
 

In [65]:
# ── Load and immediately filter to called-up players ──────────────────────────
fantasy_raw = pd.read_csv(FANTASY_CSV)
print(f"Total players in CSV:          {len(fantasy_raw)}")
print(f"Status value counts:\n{fantasy_raw['status'].value_counts().to_string()}\n")
 
fantasy = fantasy_raw[fantasy_raw["status"] == "playing"].reset_index(drop=True)
print(f"Players after status==playing filter: {len(fantasy)}")
print(f"\nPlayers per team (filtered squad):")
team_counts = fantasy.groupby("team")["name"].count().sort_values(ascending=False)
print(team_counts.to_string())
print(f"\nPositions in filtered squad:")
print(fantasy["position"].value_counts().to_string())
 

Total players in CSV:          749
Status value counts:
status
playing        623
transferred    124
injured          1
suspended        1

Players after status==playing filter: 623

Players per team (filtered squad):
team
Bosnia and Herzegovina    27
Algeria                   26
Australia                 26
Argentina                 26
Austria                   26
Belgium                   26
Brazil                    26
Cabo Verde                26
Colombia                  26
Congo DR                  26
Croatia                   26
Ecuador                   26
Ghana                     26
England                   26
Morocco                   26
Mexico                    26
Senegal                   26
Spain                     26
Norway                    26
Portugal                  26
Switzerland               26
USA                       26
Egypt                     25
Canada                    25

Positions in filtered squad:
position
MID    227
DEF    198
FWD    125
GK      7

In [66]:
# %% CELL 2 — Helper functions
 
def lev(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for ca in a:
        curr = [prev[0] + 1]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[-1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]
 
 
def norm(s) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z ]", "", re.sub(r"\s+", " ", s)).strip()
 

In [67]:
# %% CELL 3 — Load stats and aggregate

stats = pd.read_csv(STATS_CSV)
print(f"Stats rows (raw):  {len(stats)}")
print(f"Stats columns:     {list(stats.columns)}\n")

stats["fbref_code"] = stats["Nation"].str.extract(r"([A-Z]{2,4})$")

count_cols = [c for c in MERGE_COUNT_COLS if c in stats.columns]
rate_cols  = [c for c in MERGE_RATE_COLS  if c in stats.columns]
meta_cols  = [c for c in MERGE_META_COLS  if c in stats.columns]

print(f"Count cols found in stats: {len(count_cols)}/{len(MERGE_COUNT_COLS)}")
print(f"Rate cols found in stats:  {len(rate_cols)}/{len(MERGE_RATE_COLS)}")
missing_count = [c for c in MERGE_COUNT_COLS if c not in stats.columns]
if missing_count:
    print(f"Missing count cols: {missing_count}")

agg_dict = {}
for c in count_cols:
    agg_dict[c] = "sum"
for c in rate_cols:
    agg_dict[c] = "first"
for c in meta_cols:
    if c == "Squad":
        agg_dict[c] = lambda x: " / ".join(x.dropna().astype(str).unique())
    else:
        agg_dict[c] = "first"

stats_agg = (
    stats
    .groupby(["Player", "fbref_code"], as_index=False, sort=False)
    .agg(agg_dict)
)

club_counts = (
    stats.groupby(["Player", "fbref_code"])
    .size()
    .reset_index(name="_n_clubs")
)
stats_agg = stats_agg.merge(club_counts, on=["Player", "fbref_code"], how="left")
stats_agg["fbref_multi_club"] = stats_agg["_n_clubs"] > 1
stats_agg = stats_agg.drop(columns=["_n_clubs"])
stats_agg["_name_norm"] = stats_agg["Player"].apply(norm)

print(f"\nStats after aggregation: {len(stats_agg)} unique player+nation combos")
print(f"Multi-club players:      {stats_agg['fbref_multi_club'].sum()}")

Stats rows (raw):  2839
Stats columns:     ['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Rk_stats_keeper', 'Nation_stats_keeper', 'Pos_stats_keeper', 'Comp_stats_keeper', 'Age_stats_keeper', 'Born_stats_keeper', 'MP_stats_keeper', 'Starts_stats_keeper', 'Min_stats_keeper', '90s_stats_keeper', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm', 'Rk_stats_shooting', 'Nation_stats_shooting', 'Pos_stats_shooting', 'Comp_stats_shooting', 'Age_stats_shooting', 'Born_stats_shooting', '90s_stats_shooting', 'Gls_stats_shooting', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK_stats_shooting', 'PKatt_stats_shooting', 'Rk_stats_playing_time', 'Nation_stats_playing_time', 'Pos_stats_playing_time', 'Comp_stats_playing_time', 'Age_stats_playing_time', 'Born_stats_playing_time', 'MP_stats_playing_time', 'Min_s

In [68]:
# %% CELL 4 — Map fantasy teams to FBref codes

NATION_MAP = {
    "Algeria":                  "ALG",
    "Argentina":                "ARG",
    "Australia":                "AUS",
    "Austria":                  "AUT",
    "Belgium":                  "BEL",
    "Bosnia and Herzegovina":   "BIH",
    "Brazil":                   "BRA",
    "Cabo Verde":               "CPV",
    "Canada":                   "CAN",
    "Colombia":                 "COL",
    "Congo DR":                 "COD",
    "Croatia":                  "CRO",
    "Curacao":                  "CUW",
    "Czechia":                  "CZE",
    "Ecuador":                  "ECU",
    "Egypt":                    "EGY",
    "England":                  "ENG",
    "France":                   "FRA",
    "Germany":                  "GER",
    "Ghana":                    "GHA",
    "Haiti":                    "HAI",
    "IR Iran":                  "IRN",
    "Iraq":                     "IRQ",
    "Japan":                    "JPN",
    "Jordan":                   "JOR",
    "Korea Republic":           "KOR",
    "Mexico":                   "MEX",
    "Morocco":                  "MAR",
    "Netherlands":              "NED",
    "New Zealand":              "NZL",
    "Norway":                   "NOR",
    "Panama":                   "PAN",
    "Paraguay":                 "PAR",
    "Portugal":                 "POR",
    "Qatar":                    "QAT",
    "Saudi Arabia":             "KSA",
    "Scotland":                 "SCO",
    "Senegal":                  "SEN",
    "South Africa":             "RSA",
    "Spain":                    "ESP",
    "Sweden":                   "SWE",
    "Switzerland":              "SUI",
    "Tunisia":                  "TUN",
    "Turkiye":                  "TUR",
    "Uruguay":                  "URU",
    "USA":                      "USA",
    "Uzbekistan":               "UZB",
    "Cote d Ivoire":            "CIV",
}

fantasy["fbref_code"] = fantasy["team"].map(NATION_MAP)

unmapped = sorted(fantasy.loc[fantasy["fbref_code"].isna(), "team"].dropna().unique())
if unmapped:
    available_codes = sorted(stats_agg["fbref_code"].dropna().unique())
    print(f"\nUNMAPPED TEAMS — add to NATION_MAP:")
    for t in unmapped:
        print(f"    '{t}': '???',")
    print(f"\nFBref codes in stats data:\n{available_codes}")
else:
    print("✓ All fantasy teams mapped to FBref codes")

print(f"\nFantasy players per FBref code (filtered squad):")
print(fantasy.groupby("fbref_code")["name"].count().sort_values(ascending=False).to_string())



✓ All fantasy teams mapped to FBref codes

Fantasy players per FBref code (filtered squad):
fbref_code
BIH    27
ALG    26
AUS    26
ARG    26
AUT    26
BEL    26
BRA    26
COD    26
POR    26
COL    26
CPV    26
CRO    26
ECU    26
ENG    26
ESP    26
GHA    26
SUI    26
MAR    26
MEX    26
NOR    26
USA    26
SEN    26
CAN    25
EGY    25


In [69]:
# %% CELL 5 — Filter stats to tournament nations only

codes_in_tournament = set(fantasy["fbref_code"].dropna().unique())
stats_filtered = stats_agg[stats_agg["fbref_code"].isin(codes_in_tournament)].copy()

print(f"Stats rows for tournament nations: {len(stats_filtered)}")
print(f"\nFBref players available per nation:")
for code in sorted(codes_in_tournament):
    n_stats  = (stats_filtered["fbref_code"] == code).sum()
    n_fantasy = (fantasy["fbref_code"] == code).sum()
    print(f"  {code}: {n_stats:>3} in FBref stats  |  {n_fantasy:>2} in fantasy squad")



Stats rows for tournament nations: 1242

FBref players available per nation:
  ALG:  22 in FBref stats  |  26 in fantasy squad
  ARG:  74 in FBref stats  |  26 in fantasy squad
  AUS:   4 in FBref stats  |  26 in fantasy squad
  AUT:  29 in FBref stats  |  26 in fantasy squad
  BEL:  55 in FBref stats  |  26 in fantasy squad
  BIH:  10 in FBref stats  |  27 in fantasy squad
  BRA:  87 in FBref stats  |  26 in fantasy squad
  CAN:   9 in FBref stats  |  25 in fantasy squad
  COD:  13 in FBref stats  |  26 in fantasy squad
  COL:  18 in FBref stats  |  26 in fantasy squad
  CPV:   1 in FBref stats  |  26 in fantasy squad
  CRO:  29 in FBref stats  |  26 in fantasy squad
  ECU:   9 in FBref stats  |  26 in fantasy squad
  EGY:   4 in FBref stats  |  25 in fantasy squad
  ENG: 189 in FBref stats  |  26 in fantasy squad
  ESP: 401 in FBref stats  |  26 in fantasy squad
  GHA:  26 in FBref stats  |  26 in fantasy squad
  MAR:  46 in FBref stats  |  26 in fantasy squad
  MEX:   5 in FBref sta

In [70]:
# %% CELL 6 — Fuzzy match function

def find_best_match(fantasy_name: str, pool: pd.DataFrame,
                    max_dist: int = MAX_EDIT_DIST):
    fn = norm(fantasy_name)
    if not fn:
        return None, None
    best_dist, best_idx = max_dist + 1, None
    for idx, cn in zip(pool.index, pool["_name_norm"]):
        if abs(len(cn) - len(fn)) > max_dist:
            continue
        d = lev(fn, cn)
        if d < best_dist:
            best_dist, best_idx = d, idx
    if best_idx is not None and best_dist <= max_dist:
        return best_idx, best_dist
    return None, None

In [71]:
# %% CELL 7 — Run merge

cols_to_add = count_cols + rate_cols + meta_cols + ["fbref_multi_club"]
cols_to_add = [c for c in cols_to_add if c in stats_filtered.columns]

for col in cols_to_add:
    fantasy[f"fbref_{col}"] = pd.NA
fantasy["fbref_matched_player"] = pd.NA
fantasy["fbref_match_dist"]     = pd.NA

match_log = []

for code, grp in fantasy.groupby("fbref_code", dropna=True):
    pool = stats_filtered[stats_filtered["fbref_code"] == code]
    if pool.empty:
        continue
    for idx in grp.index:
        fname = fantasy.at[idx, "name"]
        match_idx, dist = find_best_match(fname, pool)
        if match_idx is None:
            continue
        row = stats_filtered.loc[match_idx]
        for col in cols_to_add:
            if col in row.index:
                fantasy.at[idx, f"fbref_{col}"] = row[col]
        fantasy.at[idx, "fbref_matched_player"] = row["Player"]
        fantasy.at[idx, "fbref_match_dist"]     = dist
        match_log.append({
            "nation":       code,
            "fantasy_name": fname,
            "fbref_name":   row["Player"],
            "dist":         dist,
            "position":     fantasy.at[idx, "position"],
        })

matched = fantasy["fbref_match_dist"].notna().sum()
total   = len(fantasy)
print(f"Matched {matched} / {total} fantasy players ({matched/total:.1%})")


Matched 312 / 623 fantasy players (50.1%)


In [72]:
# %% CELL 8 — Match quality report

match_df = pd.DataFrame(match_log)
if not match_df.empty:
    print(f"\nMatch distance breakdown:")
    print(f"  Exact  (dist=0): {(match_df['dist']==0).sum()}")
    print(f"  Near   (dist=1): {(match_df['dist']==1).sum()}")
    print(f"  Fuzzy  (dist=2): {(match_df['dist']==2).sum()}")
    print(f"  Fuzzy  (dist=3): {(match_df['dist']==3).sum()}")

    suspect = match_df[match_df["dist"] >= 1].sort_values(["dist","nation"], ascending=[False,True])
    if not suspect.empty:
        print(f"\nSuspect matches (dist ≥ 1) — review manually:")
        print(suspect[["nation","fantasy_name","fbref_name","dist","position"]].to_string(index=False))

print(f"\nMatch rate by nation (fantasy squad only):")
no_top5 = []
for code, grp in fantasy.groupby("fbref_code", dropna=True):
    n_matched = grp["fbref_match_dist"].notna().sum()
    n_total   = len(grp)
    pct       = n_matched / n_total if n_total else 0
    flag      = "  ← no top-5 league data" if n_matched == 0 else ""
    print(f"  {code}: {n_matched:>2}/{n_total:<2} ({pct:.0%}){flag}")
    if n_matched == 0:
        no_top5.append(code)

if no_top5:
    print(f"\nNations with 0 FBref matches (likely no top-5 league players): {no_top5}")

print(f"\nUnmatched fantasy players (no FBref stats found):")
unmatched = fantasy[fantasy["fbref_match_dist"].isna()][["name","team","position","price"]]
print(unmatched.groupby("position")["name"].count().rename("unmatched_count").to_string())
print(f"\nTotal unmatched: {len(unmatched)}")


Match distance breakdown:
  Exact  (dist=0): 307
  Near   (dist=1): 5
  Fuzzy  (dist=2): 0
  Fuzzy  (dist=3): 0

Suspect matches (dist ≥ 1) — review manually:
nation                fantasy_name                  fbref_name  dist position
   ALG             Mohammed Amoura              Mohamed Amoura     1      FWD
   BRA               Luiz Henrique               Luis Henrique     1      FWD
   CRO               Marco Pasalic               Mario Pašalić     1      FWD
   ESP                 Yéremy Pino                 Yeremi Pino     1      MID
   MAR Ayoube Amaimouni-Echghouyab Ayoube Amaimouni Echghouyab     1      FWD

Match rate by nation (fantasy squad only):
  ALG: 11/26 (42%)
  ARG: 18/26 (69%)
  AUS:  4/26 (15%)
  AUT: 17/26 (65%)
  BEL: 21/26 (81%)
  BIH:  7/27 (26%)
  BRA: 15/26 (58%)
  CAN:  6/25 (24%)
  COD: 10/26 (38%)
  COL:  8/26 (31%)
  CPV:  1/26 (4%)
  CRO: 18/26 (69%)
  ECU:  7/26 (27%)
  EGY:  3/25 (12%)
  ENG: 25/26 (96%)
  ESP: 24/26 (92%)
  GHA: 13/26 (50%)
  MAR:

In [73]:
# %% CELL 9 — Save

fantasy = fantasy.drop(columns=["fbref_code"], errors="ignore")
fantasy.to_csv(OUT_CSV, index=False)

print(f"\nSaved → {OUT_CSV}")
print(f"Shape: {fantasy.shape[0]} rows × {fantasy.shape[1]} cols")
fbref_cols = [c for c in fantasy.columns if c.startswith("fbref_")]
print(f"FBref columns added ({len(fbref_cols)}): {fbref_cols}")



Saved → fantasy_enriched.csv
Shape: 623 rows × 106 cols
FBref columns added (44): ['fbref_MP', 'fbref_Starts', 'fbref_Min', 'fbref_90s', 'fbref_Gls', 'fbref_Ast', 'fbref_G+A', 'fbref_G-PK', 'fbref_TklW', 'fbref_Int', 'fbref_GA', 'fbref_Saves', 'fbref_CS', 'fbref_PKA', 'fbref_PKsv', 'fbref_CrdY', 'fbref_CrdR', 'fbref_PK', 'fbref_PKatt', 'fbref_SoTA', 'fbref_PKm', 'fbref_Sh', 'fbref_SoT', 'fbref_Fls', 'fbref_Fld', 'fbref_Off', 'fbref_Crs', 'fbref_OG', 'fbref_2CrdY', 'fbref_Save%', 'fbref_CS%', 'fbref_GA90', 'fbref_SoT%', 'fbref_Sh/90', 'fbref_SoT/90', 'fbref_G/Sh', 'fbref_G/SoT', 'fbref_Pos', 'fbref_Squad', 'fbref_Comp', 'fbref_Age', 'fbref_fbref_multi_club', 'fbref_matched_player', 'fbref_match_dist']
